#Middleware

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

/home/anshul/anaconda3/envs/astro/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/anshul/anaconda3/envs/astro/lib/python3.10/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
model =ChatOllama(
    model="qwen3:0.6b",
)
model

ChatOllama(output_version=None, model='qwen3:0.6b')

In [3]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),##after that times of message it will trigger middleware
            keep=("messages",4)
        )
    ]
    
)
config={"configurable":{"thread_id":"test"}}

In [4]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]


In [5]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"message:{response}")
    print(f"message{len(response['messages'])}")
    print(response)

message:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='99ec9495-07bb-4183-b5c6-e267859fd6dc'), AIMessage(content='<think>\nOkay, so the user is asking, "What is 2+2?" and I need to figure out the answer. Let me think. First, I know that in mathematics, when you add 2 and 2, you get 4. But maybe the user is thinking of something else, like in a different context. Wait, maybe they are playing a game or using some code? Or perhaps they are thinking of something related to programming, like in a programming problem?\n\nLet me break it down. In basic arithmetic, 2 plus 2 is definitely 4. But if it\'s a different context, like a math puzzle, maybe there are some tricks here. For example, in some puzzles, numbers can be manipulated in special ways. Or maybe in a different language where the numbers are written differently? Or maybe in a non-English language, numbers are represented differently. But I\'m not sure.\n\nAlternatively, could it b

## BASED on the token 

In [6]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

In [7]:
@tool
def search_hospital(city):
    """search best hospital"""
    return f"hospital best place {city}"

In [8]:
agent = create_agent(
    model=model,
    tools=[search_hospital],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model=model,

        trigger=('tokens',400),
        keep=('tokens',200)
    )
    ]
)
config = {"configurable": {"thread_id": "test-1"}}

def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars//4


In [9]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for c in cities:
    response = agent.invoke(
        {
            "messages":[HumanMessage(content=f"find hospital in {c}")]
            
        },config=config
    )
    print(response['messages'])
    

[HumanMessage(content='find hospital in Paris', additional_kwargs={}, response_metadata={}, id='437295ba-e776-400b-b5ba-05e18ab0c27a'), AIMessage(content="<think>\nOkay, the user is asking to find a hospital in Paris. Let me check the available tools. There's a function called search_hospital that takes a city parameter. The parameters required are city, which is a required field. Since the user specified Paris, I need to call the search_hospital function with the city set to Paris. No other parameters are needed here. I should make sure to format the tool call correctly in JSON within the specified XML tags.\n</think>\n\n", additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-06-21T12:33:38.597996286Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4160800272, 'load_duration': 12307973, 'prompt_eval_count': 139, 'prompt_eval_duration': 1154233328, 'eval_count': 118, 'eval_duration': 2992441623, 'model_name': 'qwen3:0.6b'}, id='lc_run--019eea2c-0

## Human Loop

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware